In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Objectives


- Understand PySpark and Pandas: Explain the core functionalities and use cases of PySpark for big data processing and Pandas for data manipulation.


- Set Up the Environment: Install and configure PySpark and Pandas to work together in a Python environment.


- Load and Explore Data: Import data into Pandas and PySpark DataFrames and perform basic data exploration.


- Convert Between DataFrames: Convert a Pandas DataFrame to a Spark DataFrame for distributed processing.


- Perform Data Manipulation: Create new columns, filter data, and perform aggregations using PySpark.


- Utilize SQL Queries: Use Spark SQL for querying data and leveraging User-Defined Functions (UDFs).


## Setting Up the Environment


- First, Let's install the necessary libraries if they are not already installed.


In [1]:
!pip install pyspark

In [2]:
!pip install findspark

In [3]:
!pip install pandas

## Initializing a Spark Session


A Spark session is crucial for working with PySpark. It enables DataFrame creation, data loading, and various operations.


### Importing Libraries


- `findspark` is used to locate the Spark installation.
- `pandas` is imported for data manipulation.


### Creating a Spark Session


- `SparkSession.builder.appName("COVID-19 Data Analysis").getOrCreate()` initializes a Spark session with the specified application name.


### Checking Spark Session


- The code checks if the Spark session is active and prints an appropriate message.


In [4]:
import findspark  # This helps us find and use Apache Spark
findspark.init()  # Initialize findspark to locate Spark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType
import pandas as pd


In [5]:
# Initialize a Spark Session
spark = SparkSession.builder.appName("BigDataAnalysisLecture").config("spark.sql.execution.arrow.pyspark.enabled", "true").getOrCreate()

# the .appName("BigDataAnalysisLecture") function sets the name of the Spark application. This name will appear in Spark's UI, logs, and cluster
# managers (like YARN or Mesos) to identify this specific application. It is useful for monitoring and managing Spark applications, especially if
# multiple applications are running on the same cluster.

# .config("spark.sql.execution.arrow.pyspark.enabled", "true") is optional and we only need it if you plan to leverage Pandas within Spark for better
# performance when converting between PySpark and Pandas. Enabling this configuration allows Apache Arrow to optimize data exchange between Pandas and PySpark.
# Arrow improves the performance of operations that involve data conversion between PySpark and Pandas (e.g., toPandas() and fromPandas()) by using in-memory columnar data structures.
# Read more about Arrow here: https://arrow.apache.org/


# Check if the Spark Session is active
if 'spark' in locals() and isinstance(spark, SparkSession):
    print("SparkSession is active and ready to use.")
else:
    print("SparkSession is not active. Please create a SparkSession.")

SparkSession is active and ready to use.


## Importing Data into Pandas from Various Sources


Let's read the COVID-19 data from the provided URL.


In [7]:
# Read the COVID-19 data from the provided URL
vaccination_data = pd.read_csv('Third_notebook_MoreIntroductionSpark_vaccination_data.csv')

##  Displaying the First Five Records


### To retrieve and print the first five records


- `vaccination_data.head()` retrieves the first five rows of the DataFrame vaccination_data.This gives us a quick look at the data contained within the dataset.
- The `print` function is used to display a message indicating what is being shown, followed by the actual data.


In [8]:
vaccination_data.head()

,iso_code,continent,location,last_updated_date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2024-08-04,235214.0,0.0,0.000,7998.0,0.0,0.0,...,NaN,37.746,0.50,64.83,0.511,4.112877e+07,NaN,NaN,NaN,NaN
1,OWID_AFR,NaN,Africa,2024-08-04,13145380.0,36.0,5.143,259117.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,1.426737e+09,NaN,NaN,NaN,NaN
2,ALB,Europe,Albania,2024-08-04,335047.0,0.0,0.000,3605.0,0.0,0.0,...,51.2,NaN,2.89,78.57,0.795,2.842318e+06,NaN,NaN,NaN,NaN
3,DZA,Africa,Algeria,2024-08-04,272139.0,18.0,2.571,6881.0,0.0,0.0,...,30.4,83.741,1.90,76.88,0.748,4.490323e+07,NaN,NaN,NaN,NaN
4,ASM,Oceania,American Samoa,2024-08-04,8359.0,0.0,0.000,34.0,0.0,0.0,...,NaN,NaN,NaN,73.74,NaN,4.429500e+04,NaN,NaN,NaN,NaN


In [9]:
vaccination_data.shape

(247, 67)

In [10]:
vaccination_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247 entries, 0 to 246
Data columns (total 67 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   iso_code                                    247 non-null    object 
 1   continent                                   235 non-null    object 
 2   location                                    247 non-null    object 
 3   last_updated_date                           247 non-null    object 
 4   total_cases                                 246 non-null    float64
 5   new_cases                                   242 non-null    float64
 6   new_cases_smoothed                          242 non-null    float64
 7   total_deaths                                246 non-null    float64
 8   new_deaths                                  243 non-null    float64
 9   new_deaths_smoothed                         243 non-null    float64
 10  total_cases_pe

### Selecting Specific Columns:


- Let\'s define a list called `columns_to_display`, which contains the names of the columns as : `['continent', 'total_cases', 'total_deaths', 'total_vaccinations', 'population']`.
- By using `vaccination_data[columns_to_display].head()`, let\'s filter the DataFrame to only show the specified columns and again display the first five records of this subset.
- The continent column is explicitly converted to string, while the numeric columns (total cases, total deaths, total vaccinations, population) are filled with zeros for NaN values and then converted to int64 (which is compatible with LongType in Spark).
- The use of fillna(0) ensures that NaN values do not cause type issues during the Spark DataFrame creation.


In [11]:
print("Displaying the first 5 records of the vaccination data:")
columns_to_display = ['continent', 'total_cases', 'total_deaths', 'total_vaccinations', 'population']
# Show the first 5 records
sub_df = vaccination_data[columns_to_display]

Displaying the first 5 records of the vaccination data:


In [12]:
sub_df

,continent,total_cases,total_deaths,total_vaccinations,population
0,Asia,235214.0,7998.0,NaN,4.112877e+07
1,NaN,13145380.0,259117.0,NaN,1.426737e+09
2,Europe,335047.0,3605.0,NaN,2.842318e+06
3,Africa,272139.0,6881.0,NaN,4.490323e+07
4,Oceania,8359.0,34.0,NaN,4.429500e+04
...,...,...,...,...,...
242,Oceania,3760.0,9.0,NaN,1.159600e+04
243,NaN,775866783.0,7057132.0,1.357877e+10,7.975105e+09
244,Asia,11945.0,2159.0,NaN,3.369661e+07
245,Africa,349842.0,4077.0,NaN,2.001767e+07


##  Converting the Pandas DataFrame to a Spark DataFrame


In [13]:
sub_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247 entries, 0 to 246
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   continent           235 non-null    object 
 1   total_cases         246 non-null    float64
 2   total_deaths        246 non-null    float64
 3   total_vaccinations  17 non-null     float64
 4   population          247 non-null    float64
dtypes: float64(4), object(1)
memory usage: 9.8+ KB


Let\'s convert the Pandas DataFrame, which contains our COVID-19 vaccination data, into a Spark DataFrame. This conversion is crucial as it allows us to utilize Spark\'s distributed computing capabilities, enabling us to handle larger datasets and perform operations in a more efficient manner.


### Defining the Schema:


- **StructType**:
  - A class that defines a structure for a DataFrame.

- **StructField**:
  - Represents a single field in the schema.
  - **Parameters**:
    1. **Field Name**: The name of the field.
    2. **Data Type**: The type of data for the field.
    3. **Nullable**: A boolean indicating whether null values are allowed.

- **Data Types**:
  - **StringType()**: Used for text fields.
  - **LongType()**: Used for numerical fields.


### Data Type Conversion:


- **astype(str)**:
  - Used to convert the `'continent'` column to string type.

- **fillna(0)**:
  - Replaces any NaN values with 0, ensuring that the numerical fields do not contain any missing data.

- **astype('int64')**:
  - Converts the columns from potentially mixed types to 64-bit integers for consistent numerical representation.


### Creating a Spark DataFrame:


- **createDataFrame**:
  - The `createDataFrame` method of the Spark session (`spark`) is called.
  - **Parameters**:
    - It takes as input a subset of the pandas DataFrame that corresponds to the fields defined in the schema, accessed using `schema.fieldNames()`.
- This function automatically converts the Pandas DataFrame into a Spark DataFrame, which is designed to handle larger datasets across a distributed environment.


- The resulting spark_df will have the defined schema, which ensures consistency and compatibility with Spark's data processing capabilities.


### Storing the Result:


In [14]:
# Convert to Spark DataFrame directly
# Define the schema
schema = StructType([
    StructField("continent", StringType(), True),
    StructField("total_cases", LongType(), True),
    StructField("total_deaths", LongType(), True),
    StructField("total_vaccinations", LongType(), True),
    StructField("population", LongType(), True)
])



In [15]:
schema

StructType([StructField('continent', StringType(), True), StructField('total_cases', LongType(), True), StructField('total_deaths', LongType(), True), StructField('total_vaccinations', LongType(), True), StructField('population', LongType(), True)])

In [16]:
# Convert the columns to the appropriate data types
sub_df['continent'] = sub_df['continent'].astype(str)  # Ensures continent is a string
sub_df['total_cases'] = sub_df['total_cases'].fillna(0).astype('int64')  # Fill NaNs and convert to int
sub_df['total_deaths'] = sub_df['total_deaths'].fillna(0).astype('int64')  # Fill NaNs and convert to int
sub_df['total_vaccinations'] = sub_df['total_vaccinations'].fillna(0).astype('int64')  # Fill NaNs and convert to int
sub_df['population'] = sub_df['population'].fillna(0).astype('int64')  # Fill NaNs and convert to int



/tmp/ipykernel_989/946790899.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['continent'] = sub_df['continent'].astype(str)  # Ensures continent is a string
/tmp/ipykernel_989/946790899.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['total_cases'] = sub_df['total_cases'].fillna(0).astype('int64')  # Fill NaNs and convert to int
/tmp/ipykernel_989/946790899.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_i

In [17]:
sub_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247 entries, 0 to 246
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   continent           247 non-null    object
 1   total_cases         247 non-null    int64 
 2   total_deaths        247 non-null    int64 
 3   total_vaccinations  247 non-null    int64 
 4   population          247 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 9.8+ KB


In [18]:
sub_df

,continent,total_cases,total_deaths,total_vaccinations,population
0,Asia,235214,7998,0,41128772
1,nan,13145380,259117,0,1426736614
2,Europe,335047,3605,0,2842318
3,Africa,272139,6881,0,44903228
4,Oceania,8359,34,0,44295
...,...,...,...,...,...
242,Oceania,3760,9,0,11596
243,nan,775866783,7057132,13578774356,7975105024
244,Asia,11945,2159,0,33696612
245,Africa,349842,4077,0,20017670


In [19]:
schema.fieldNames()

['continent',
 'total_cases',
 'total_deaths',
 'total_vaccinations',
 'population']

In [20]:
spark_df = spark.createDataFrame(sub_df[schema.fieldNames()])  # Use only the specified fields
# Show the Spark DataFrame
spark_df.show(5)

+---------+-----------+------------+------------------+----------+
|continent|total_cases|total_deaths|total_vaccinations|population|
+---------+-----------+------------+------------------+----------+
|     Asia|     235214|        7998|                 0|  41128772|
|      nan|   13145380|      259117|                 0|1426736614|
|   Europe|     335047|        3605|                 0|   2842318|
|   Africa|     272139|        6881|                 0|  44903228|
|  Oceania|       8359|          34|                 0|     44295|
+---------+-----------+------------+------------------+----------+
only showing top 5 rows


## Checking the Structure of the Spark DataFrame


In this section, Let\'s examine the structure of the Spark DataFrame that we created from the Pandas DataFrame. Understanding the schema of a DataFrame is crucial as it provides insight into the data types of each column and helps ensure that the data is organized correctly for analysis.


### Displaying the Schema:

- The method `spark_df.printSchema()` is called to output the structure of the Spark DataFrame.
- This method prints the names of the columns along with their data types (e.g., `StringType`, `IntegerType`, `DoubleType`, etc.), providing a clear view of how the data is organized.


In [21]:
print("Schema of the Spark DataFrame:")
spark_df.printSchema()
# Print the structure of the DataFrame (columns and types)

# nullable = true indicates that the column can contain null (missing) values. This means that Spark will allow null entries in that column

Schema of the Spark DataFrame:
root
 |-- continent: string (nullable = true)
 |-- total_cases: long (nullable = true)
 |-- total_deaths: long (nullable = true)
 |-- total_vaccinations: long (nullable = true)
 |-- population: long (nullable = true)



##  Basic Data Exploration


In this section, let\'s perform basic data exploration on the Spark DataFrame. This step is essential for understanding the dataset better, allowing us to gain insights and identify any patterns or anomalies. Let\'s demonstrate how to view specific contents of the DataFrame, select certain columns, and filter records based on conditions.


### Picking Specific Columns


- To display only certain columns,let\'s use the following code:


In [22]:
print("Displaying the 'continent' and 'total_cases' columns:")
# Show only the 'continent' and 'total_cases' columns
spark_df.select('continent', 'total_cases').show(5)

Displaying the 'continent' and 'total_cases' columns:
+---------+-----------+
|continent|total_cases|
+---------+-----------+
|     Asia|     235214|
|      nan|   13145380|
|   Europe|     335047|
|   Africa|     272139|
|  Oceania|       8359|
+---------+-----------+
only showing top 5 rows


###  Sifting Through Data

- To filter records based on a specific condition, let\'s use the following code:


In [23]:
print("Filtering records where 'total_cases' is greater than 1,000,000:")
 # Show records with more than 1 million total cases
spark_df.filter(spark_df['total_cases'] > 1000000).show(5)

Filtering records where 'total_cases' is greater than 1,000,000:
+-------------+-----------+------------+------------------+----------+
|    continent|total_cases|total_deaths|total_vaccinations|population|
+-------------+-----------+------------+------------------+----------+
|          nan|   13145380|      259117|                 0|1426736614|
|South America|   10101218|      130663|                 0|  45510324|
|          nan|  301499099|     1637249|        9104304615|4721383370|
|      Oceania|   11861161|       25236|                 0|  26177410|
|       Europe|    6082444|       22534|                 0|   8939617|
+-------------+-----------+------------+------------------+----------+
only showing top 5 rows


##  Working with Columns


In this section, let\'s create a new column called `death_percentage`, which calculates the death rate during the COVID-19 pandemic. This calculation is based on the total_deaths (the count of deaths) and the population (the total population) columns in our Spark DataFrame. This new metric will provide valuable insight into the impact of COVID-19 in different regions.


- Let\'s import the functions module from `pyspark.sql` as `F`, which contains built-in functions for DataFrame operations.


### Calculating the Death Percentage:


- Let\'s create a new DataFrame `spark_df_with_percentage` by using the `withColumn()` method to add a new column called `death_percentage`.
- The formula `(spark_df['total_deaths'] / spark_df['population']) * 100` computes the death percentage by dividing the total deaths by the total population and multiplying by 100.


In [24]:
from pyspark.sql import functions as F

spark_df_with_percentage = spark_df.withColumn('death_percentage',(spark_df['total_deaths'] / spark_df['population']) * 100)

spark_df_with_percentage.show(5)

+---------+-----------+------------+------------------+----------+--------------------+
|continent|total_cases|total_deaths|total_vaccinations|population|    death_percentage|
+---------+-----------+------------+------------------+----------+--------------------+
|     Asia|     235214|        7998|                 0|  41128772| 0.01944624069981958|
|      nan|   13145380|      259117|                 0|1426736614|0.018161516110078605|
|   Europe|     335047|        3605|                 0|   2842318|  0.1268330989002638|
|   Africa|     272139|        6881|                 0|  44903228|0.015324065343364624|
|  Oceania|       8359|          34|                 0|     44295| 0.07675809910825149|
+---------+-----------+------------+------------------+----------+--------------------+
only showing top 5 rows


### Formatting the Percentage:


- Let\'s update the `death_percentage` column to format its values to two decimal places using `F.format_number()`, and concatenate a percentage symbol using `F.concat()` and `F.lit('%')`.
- This makes the death percentage easier to read and interpret.


In [25]:
spark_df_with_percentage = spark_df_with_percentage.withColumn(
    'death_percentage',
    F.concat(
        # Format to 2 decimal places
        F.format_number(spark_df_with_percentage['death_percentage'], 2),
        # Append the percentage symbol
        F.lit('%')
    )
)


In [26]:
spark_df_with_percentage.show(5)

+---------+-----------+------------+------------------+----------+----------------+
|continent|total_cases|total_deaths|total_vaccinations|population|death_percentage|
+---------+-----------+------------+------------------+----------+----------------+
|     Asia|     235214|        7998|                 0|  41128772|           0.02%|
|      nan|   13145380|      259117|                 0|1426736614|           0.02%|
|   Europe|     335047|        3605|                 0|   2842318|           0.13%|
|   Africa|     272139|        6881|                 0|  44903228|           0.02%|
|  Oceania|       8359|          34|                 0|     44295|           0.08%|
+---------+-----------+------------+------------------+----------+----------------+
only showing top 5 rows


### Selecting Relevant Columns:


- Let\'s define a list `columns_to_display` that includes `'total_deaths', 'population', 'death_percentage', 'continent', 'total_vaccinations', and 'total_cases'`.
- Finally, let's display the first five records of the modified DataFrame with the new column by calling `spark_df_with_percentage.select(columns_to_display).show(5)`.


In [27]:
columns_to_display = ['total_deaths', 'population', 'death_percentage', 'continent', 'total_vaccinations', 'total_cases']
spark_df_with_percentage.select(columns_to_display).show(5)

+------------+----------+----------------+---------+------------------+-----------+
|total_deaths|population|death_percentage|continent|total_vaccinations|total_cases|
+------------+----------+----------------+---------+------------------+-----------+
|        7998|  41128772|           0.02%|     Asia|                 0|     235214|
|      259117|1426736614|           0.02%|      nan|                 0|   13145380|
|        3605|   2842318|           0.13%|   Europe|                 0|     335047|
|        6881|  44903228|           0.02%|   Africa|                 0|     272139|
|          34|     44295|           0.08%|  Oceania|                 0|       8359|
+------------+----------+----------------+---------+------------------+-----------+
only showing top 5 rows


In [28]:
spark_df_with_percentage.show()

+-------------+-----------+------------+------------------+----------+----------------+
|    continent|total_cases|total_deaths|total_vaccinations|population|death_percentage|
+-------------+-----------+------------+------------------+----------+----------------+
|         Asia|     235214|        7998|                 0|  41128772|           0.02%|
|          nan|   13145380|      259117|                 0|1426736614|           0.02%|
|       Europe|     335047|        3605|                 0|   2842318|           0.13%|
|       Africa|     272139|        6881|                 0|  44903228|           0.02%|
|      Oceania|       8359|          34|                 0|     44295|           0.08%|
|       Europe|      48015|         159|                 0|     79843|           0.20%|
|       Africa|     107481|        1937|                 0|  35588996|           0.01%|
|North America|       3904|          12|                 0|     15877|           0.08%|
|North America|       9106|     

## Grouping and Summarizing


 Let\'s calculate the total number of deaths per continent using the data in our Spark DataFrame. Grouping and summarizing data is a crucial aspect of data analysis, as it allows us to aggregate information and identify trends across different categories.


 ### Grouping the Data


The `spark_df.groupby(['continent'])` method groups the data by the `continent` column. This means that all records associated with each continent will be aggregated together.


### Aggregating the Deaths


The `agg({"total_deaths": "SUM"})` function is used to specify the aggregation operation. In this case, we want to calculate the sum of the `total_deaths` for each continent. This operation will create a new DataFrame where each continent is listed alongside the total number of deaths attributed to it.


### Displaying the Results


The `show()` method is called to display the results of the aggregation. This will output the total number of deaths for each continent in a tabular format.


In [29]:
print("Calculating the total deaths per continent:")
# Group by continent and sum total death rates
spark_df.groupby(['continent']).agg({"total_deaths": "SUM"}).show()

Calculating the total deaths per continent:
+-------------+-----------------+
|    continent|sum(total_deaths)|
+-------------+-----------------+
|       Europe|          2102483|
|       Africa|           259117|
|          nan|         22430618|
|North America|          1671178|
|South America|          1354187|
|      Oceania|            32918|
|         Asia|          1637249|
+-------------+-----------------+



In [30]:
spark_df_with_percentage.groupby(['continent']).agg({"total_deaths": "SUM"}).show()

+-------------+-----------------+
|    continent|sum(total_deaths)|
+-------------+-----------------+
|       Europe|          2102483|
|       Africa|           259117|
|          nan|         22430618|
|North America|          1671178|
|South America|          1354187|
|      Oceania|            32918|
|         Asia|          1637249|
+-------------+-----------------+



## Exploring User-Defined Functions (UDFs)



User-Defined Functions (UDFs) in PySpark allow us to create custom functions that can be applied to individual columns within a DataFrame. This feature provides increased flexibility and customization in data processing, enabling us to define specific transformations or calculations that are not available through built-in functions. In this section, let\'s define a UDF to convert total deaths in the dataset.


### Importing pandas_udf


The `pandas_udf` function is imported from `pyspark.sql.functions`. This decorator allows us to define a UDF that operates on Pandas Series


### Defining the UDF


This function `convert_total_deaths()` takes in a parameter `total_deaths` and returns double its value. You can replace the logic with any transformation you want to apply to the column data.



### Registering the UDF


The line `spark.udf.register("convert_total_deaths", convert_total_deaths, IntegerType())` registers the UDF with Spark indicating that the function returns an integer, allowing us to use it in Spark SQL queries and DataFrame operations.


In [31]:
from pyspark.sql.types import IntegerType

# Function definition
def convert_total_deaths(total_deaths):
    return total_deaths * 5
# Here you can define any transformation you want
# Register the UDF with Spark
spark.udf.register("convert_total_deaths", convert_total_deaths, IntegerType())

<function __main__.convert_total_deaths(total_deaths)>

In [32]:
# Pandas Option
# Apply UDF to 'total_deaths' column and create a new column 'converted_total_deaths_pd'
spark_df.withColumn("converted_total_deaths_pd", convert_total_deaths(spark_df["total_deaths"])).show()


+-------------+-----------+------------+------------------+----------+-------------------------+
|    continent|total_cases|total_deaths|total_vaccinations|population|converted_total_deaths_pd|
+-------------+-----------+------------+------------------+----------+-------------------------+
|         Asia|     235214|        7998|                 0|  41128772|                    39990|
|          nan|   13145380|      259117|                 0|1426736614|                  1295585|
|       Europe|     335047|        3605|                 0|   2842318|                    18025|
|       Africa|     272139|        6881|                 0|  44903228|                    34405|
|      Oceania|       8359|          34|                 0|     44295|                      170|
|       Europe|      48015|         159|                 0|     79843|                      795|
|       Africa|     107481|        1937|                 0|  35588996|                     9685|
|North America|       3904|   

In [33]:
def user_percentage(total_cases, population):
    return total_cases/population * 100
# Here you can define any transformation you want
# Register the UDF with Spark
spark.udf.register("user_percentage", user_percentage, IntegerType())

<function __main__.user_percentage(total_cases, population)>

In [34]:
# Pandas Option
# Apply UDF to 'total_deaths' column and create a new column 'converted_total_deaths_pd'
spark_df_with_percentage.withColumn("user_%", user_percentage(spark_df_with_percentage["total_cases"],spark_df_with_percentage["population"])).show()

+-------------+-----------+------------+------------------+----------+----------------+------------------+
|    continent|total_cases|total_deaths|total_vaccinations|population|death_percentage|            user_%|
+-------------+-----------+------------+------------------+----------+----------------+------------------+
|         Asia|     235214|        7998|                 0|  41128772|           0.02%|0.5718964816163244|
|          nan|   13145380|      259117|                 0|1426736614|           0.02%|0.9213599672854544|
|       Europe|     335047|        3605|                 0|   2842318|           0.13%|11.787808401452617|
|       Africa|     272139|        6881|                 0|  44903228|           0.02%| 0.606056651428267|
|      Oceania|       8359|          34|                 0|     44295|           0.08%|18.871204424878655|
|       Europe|      48015|         159|                 0|     79843|           0.20%|  60.1367684080007|
|       Africa|     107481|        19

## Using Spark SQL


Spark SQL enables us to execute SQL queries directly on DataFrames.


In [35]:
# Drop the existing temporary view if it exists
spark.sql("DROP VIEW IF EXISTS data_v")



DataFrame[]

In [36]:
# Create a new temporary view
spark_df.createTempView('data_v')

# The createTempView('data_v') method creates an in-memory, temporary view called data_v based on the spark_df DataFrame.

# This allows you to run SQL queries directly on data_v using spark.sql() as if it were a table in a database.

# The view exists only for the duration of the Spark session and is not persisted to disk. Once the session ends, the view is removed.

## Running SQL Queries


In this step, Let\'s execute SQL queries to retrieve specific records from the temporary view which was created earlier.Let\'s demonstrate how to display all records from the data table and filter those records based on vaccination totals. This capability allows for efficient data exploration and analysis using SQL syntax.


The first query retrieves all records from the temporary view data using the SQL command SELECT * FROM data_v. The show() method is called to display the results in a tabular format. This is useful for getting an overview of the entire dataset.


In [37]:
spark.sql('SELECT * FROM data_v').show()

+-------------+-----------+------------+------------------+----------+
|    continent|total_cases|total_deaths|total_vaccinations|population|
+-------------+-----------+------------+------------------+----------+
|         Asia|     235214|        7998|                 0|  41128772|
|          nan|   13145380|      259117|                 0|1426736614|
|       Europe|     335047|        3605|                 0|   2842318|
|       Africa|     272139|        6881|                 0|  44903228|
|      Oceania|       8359|          34|                 0|     44295|
|       Europe|      48015|         159|                 0|     79843|
|       Africa|     107481|        1937|                 0|  35588996|
|North America|       3904|          12|                 0|     15877|
|North America|       9106|         146|                 0|     93772|
|South America|   10101218|      130663|                 0|  45510324|
|         Asia|     452273|        8777|                 0|   2780472|
|North

### Filtering Records


The second query is designed to filter the dataset to show only those continents where the total vaccinations exceed 1 million. The SQL command used here is `SELECT continent FROM data_v WHERE total_vaccinations > 1000000`. The `show()` method is again used to display the results, specifically listing the continents that meet the filter criteria.


In [38]:
print("Displaying continent with total vaccinated more than 1 million:")
# SQL filtering
spark.sql("SELECT continent FROM data_v WHERE total_vaccinations > 1000000").show()

Displaying continent with total vaccinated more than 1 million:
+-------------+
|    continent|
+-------------+
|          nan|
|North America|
|       Europe|
|       Europe|
|          nan|
|          nan|
|          nan|
|         Asia|
|         Asia|
|       Europe|
|          nan|
|         Asia|
|      Oceania|
|          nan|
|          nan|
|          nan|
|          nan|
+-------------+

